In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, f1_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
from experiment_runner import load_dataset
from imputation import imputation

np.random.seed(42)

In [2]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

In [3]:
datasets = ["housing", "adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

# глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

ratios = [0.05, 0.20, 0.50]

mechanisms = ["MCAR", "MAR"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
    ],
    "KNN hybrid": [
        {"n_neighbors": 7},
    ],
    "MICE hybrid": [
        {"max_iter": 10},
    ],
    "MissForest": [
        {"n_estimators": 100},
    ]
}

In [4]:
# Функция для вычисления метрик моделей
def evaluate_model(y_true, y_pred, task_type="regression"):
    if task_type == "regression":
        rmse = root_mean_squared_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        return {"rmse": rmse, "r2": r2}
    else:
        accuracy = (y_true == y_pred).mean()
        f1 = f1_score(y_true, y_pred, average="weighted")
        metrics = {"accuracy": accuracy, "f1": f1}
        return metrics

# Словари с моделями для разных задач
models = {
    "housing": {
        "task_type": "regression",
        "models": {
            "Ridge": Ridge(random_state=42),
            "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        }
    },
    "adult": {
        "task_type": "classification",
        "models": {
            "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
            "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        }
    }
}


In [5]:
COMET_EVAL_PROJECT = "imputation-models-evaluation"

for dataset_name in datasets:
    # загружаем данные и таргет
    df_true = load_dataset(dataset_name)
    target_col = global_target[dataset_name]
    
    task_type = models[dataset_name]["task_type"]
    current_models = models[dataset_name]["models"]
    
    print(f"\n[{dataset_name.upper()}] Task: {task_type}")
    
    # сохраняем таргет
    y_true_target = df_true[target_col]
    # удаляем таргет из признаков, чтобы методы импутации не видели его
    X_true_features_raw = df_true.drop(columns=[target_col])

    # обучаем модели на оригинальных данных без пропусков (Бейзлайн)
    X_true_features = pd.get_dummies(X_true_features_raw, drop_first=True)
    
    X_tr_true, X_te_true, y_tr_true, y_te_true = train_test_split(
        X_true_features, y_true_target, test_size=0.2, random_state=42
    )
    
    # перебираем механизмы пропусков
    for mechanism in mechanisms:
        
        for method, param_list in EXPERIMENT_CONFIG.items():
            for params in param_list:
                params_str = "_".join([f"{k}={v}" for k, v in params.items()])
                
                for ml_model_name, ml_model in current_models.items():
                    
                    # для комбинации метод импутации + ML модель открываем эксперимент один раз
                    experiment = Experiment(
                        api_key=os.environ.get("COMET_API_KEY"),
                        project_name=COMET_EVAL_PROJECT,
                        workspace=None
                    )
                    
                    exp_name = f"{dataset_name} | {mechanism} | {method} | ML: {ml_model_name}"
                    experiment.set_name(exp_name)
                    
                    experiment.log_parameters({
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "imputation_method": method,
                        "ml_model": ml_model_name,
                        **(params or {})
                    })
                    
                    # логируем бейзлайн (Ratio = 0%) как первую точку в этом графике
                    ml_model.fit(X_tr_true, y_tr_true)
                    y_pred_base = ml_model.predict(X_te_true)
                    metrics_base = evaluate_model(y_te_true, y_pred_base, task_type)
                    
                    for met_name, met_val in metrics_base.items():
                        experiment.log_metric(met_name, met_val, step=0) 
                    
                    for ratio in ratios:
                        print(f"[{dataset_name}] {mechanism} | {method} | {ml_model_name} | Ratio: {ratio}")
                        
                        # генерируем пропуски
                        df_incomplete, mask = generate_missing_data(
                            data=X_true_features_raw, 
                            columns_config=columns_target[dataset_name],
                            mechanism=mechanism, 
                            ratio=ratio
                        )

                        y = y_true_target
                        
                        # восстанавливаем
                        df_imputed = imputation(
                            df_incomplete=df_incomplete,
                            algo=method,
                            **params
                        )
                        
                        # подготовка данных
                        X_imputed = pd.get_dummies(df_imputed, drop_first=True)
                        X_imputed = X_imputed.reindex(columns=X_true_features.columns, fill_value=0)
                        
                        # разделяем
                        X_train, X_test, y_train, y_test = train_test_split(
                            X_imputed, y, test_size=0.2, random_state=42
                        )
                        
                        # обучение и предсказание
                        ml_model.fit(X_train, y_train)
                        y_pred = ml_model.predict(X_test)
                        
                        # оценка
                        metrics = evaluate_model(y_test, y_pred, task_type)
                        
                        step = int(ratio * 100)
                        for met_name, met_val in metrics.items():
                            experiment.log_metric(met_name, met_val, step=step)
                            
                    # закрываем эксперимент только когда прошли все ratios
                    experiment.end()
                    print(f"-> Finished logging all ratios for {exp_name}")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.



[HOUSING] Task: regression


COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/imputation-models-evaluation/b9652551e776468ba1e60cd0f34814ab

COMET INFO: The process of logging environment details (conda environment, git patch) is underway. Please be patient as this may take some time.


[housing] MCAR | Simple | Ridge | Ratio: 0.05
[housing] MCAR | Simple | Ridge | Ratio: 0.2
[housing] MCAR | Simple | Ridge | Ratio: 0.5


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : housing | MCAR | Simple | ML: Ridge
COMET INFO:     url                   : https://www.comet.com/kam2607kam/imputation-models-evaluation/b9652551e776468ba1e60cd0f34814ab
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     r2 [4]   : (0.2678679981482084, 0.652030892209707)
COMET INFO:     rmse [4] : (67614.21114540116, 98075.9076291478)
COMET INFO:   Others:
COMET INFO:     Name : housing | MCAR | Simple | ML: Ridge
COMET INFO:   Parameters:
COMET INFO:     add_indicator       : False
COMET INFO:     copy                : True
COMET INFO:     dataset             : housing
COMET INFO:     fill_value          : None
COMET INFO:     imputation_met

...

-> Finished logging all ratios for adult | MAR | MissForest | ML: LogisticRegression


COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/imputation-models-evaluation/aa51c4118a9441d9831ddc2f830a0eb4



[adult] MAR | MissForest | RandomForest | Ratio: 0.05
[adult] MAR | MissForest | RandomForest | Ratio: 0.2
[adult] MAR | MissForest | RandomForest | Ratio: 0.5


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : adult | MAR | MissForest | ML: RandomForest
COMET INFO:     url                   : https://www.comet.com/kam2607kam/imputation-models-evaluation/aa51c4118a9441d9831ddc2f830a0eb4
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     accuracy [4] : (0.824, 0.837)
COMET INFO:     f1 [4]       : (0.8195467684755933, 0.8327810717802236)
COMET INFO:   Others:
COMET INFO:     Name : adult | MAR | MissForest | ML: RandomForest
COMET INFO:   Parameters:
COMET INFO:     dataset           : adult
COMET INFO:     imputation_method : MissForest
COMET INFO:     mechanism         : MAR
COMET INFO:     ml_model          : RandomForest
COMET INFO:     n_estimato

-> Finished logging all ratios for adult | MAR | MissForest | ML: RandomForest
